SMS SPAM CLASSIFIER - MODEL COMPARISON

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)
import joblib

LOAD DATASET

In [ ]:
df = pd.read_csv(
    "spam.csv",
    encoding="latin-1"
)

# Keep only useful columns
df = df[['v1', 'v2']]

df.columns = ['label', 'message']

print(df.head())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


PREPROCESS LABELS

In [ ]:
df['label'] = df['label'].map({
    'ham':0,
    'spam':1
})

X = df['message']
y = df['label']

TRAIN TEST SPLIT

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

DEFINE MODELS

In [ ]:
models = {

    "Naive Bayes":
    MultinomialNB(),

    "Logistic Regression":
    LogisticRegression(max_iter=1000),

    "Linear SVM":
    LinearSVC(),

    "Random Forest":
    RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ),

    "Gradient Boosting":
    GradientBoostingClassifier(
        n_estimators=100,
        random_state=42
    )
}
results = []

best_model = None
best_f1 = 0

TRAIN AND EVALUATE

In [ ]:
for name, model in models.items():

    pipeline = Pipeline([
        ("tfidf",
         TfidfVectorizer(
             stop_words='english'
         )),
        ("classifier", model)
    ])
    pipeline.fit(X_train, y_train)

    preds = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, preds)
    precision = precision_score(y_test, preds)
    recall = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)

    results.append([
        name,
        accuracy,
        precision,
        recall,
        f1
    ])
    print("\n")
    print("="*50)
    print(name)
    print("="*50)

    print(classification_report(
        y_test,
        preds
    ))

    if f1 > best_f1:
        best_f1 = f1
        best_model = pipeline
        best_model_name = name




Naive Bayes
              precision    recall  f1-score   support

           0       0.97      1.00      0.98       966
           1       1.00      0.77      0.87       149

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.97      1115



Logistic Regression
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       966
           1       1.00      0.76      0.86       149

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.97      1115



Linear SVM
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       966
           1       0.99      0.89      0.94       149

    accuracy                           0.98      1115
   macro avg       0.99      0.94      0.96      1115
weighted avg       0.98  

DISPLAY RESULTS

In [ ]:
results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1"
    ]
)

results_df = results_df.sort_values(
    by="F1",
    ascending=False
)

print("\n\nMODEL COMPARISON")
print(results_df)



MODEL COMPARISON
                 Model  Accuracy  Precision    Recall        F1
2           Linear SVM  0.983857   0.992481  0.885906  0.936170
3        Random Forest  0.974888   1.000000  0.812081  0.896296
0          Naive Bayes  0.968610   1.000000  0.765101  0.866920
1  Logistic Regression  0.967713   1.000000  0.758389  0.862595
4    Gradient Boosting  0.961435   0.990741  0.718121  0.832685


SAVE BEST MODEL

In [ ]:
print("\nBest Model:", best_model_name)
print("Best F1 Score:", best_f1)

joblib.dump(
    best_model,
    "best_spam_classifier.pkl"
)

print(
    "\nSaved as best_spam_classifier.pkl"
)


Best Model: Linear SVM
Best F1 Score: 0.9361702127659575

Saved as best_spam_classifier.pkl


TEST PREDICTIONS

In [ ]:
sample_messages = [

    "Congratulations! You won a free iPhone. Click now!",

    "Can we meet tomorrow for lunch?",

    "URGENT! Claim your cash reward now",

    "Hey, are you coming to class today?"
]

for msg in sample_messages:

    prediction = best_model.predict([msg])[0]

    label = "SPAM" if prediction == 1 else "HAM"

    print("\nMessage:", msg)
    print("Prediction:", label)


Message: Congratulations! You won a free iPhone. Click now!
Prediction: HAM

Message: Can we meet tomorrow for lunch?
Prediction: HAM

Message: URGENT! Claim your cash reward now
Prediction: SPAM

Message: Hey, are you coming to class today?
Prediction: HAM
